In [1]:
# ============================================================
# Feature Research — Explored Features
# Seoul National University | Seongwon Song
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier

# Data
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
df_sp500 = pd.read_csv(url)
tickers = df_sp500["Symbol"].tolist()[:50]  # 50개로 테스트

data = yf.download(tickers, start="2020-01-01", end="2026-08-29", auto_adjust=True)["Close"]
volume = yf.download(tickers, start="2020-01-01", end="2026-08-29", auto_adjust=True)["Volume"]
data = data.dropna(axis=1, thresh=int(len(data) * 0.8))
volume = volume[data.columns]

# ============================================================
# 1. Technical Features (기술적 지표)
# ============================================================

features = pd.DataFrame()
for ticker in data.columns:
    close = data[ticker]
    vol = volume[ticker]
    temp = pd.DataFrame(index=close.index)
    
    # Momentum
    temp["mom1w"] = close.pct_change(5)
    temp["mom1m"] = close.pct_change(21)
    temp["mom3m"] = close.pct_change(63)
    
    # Moving averages
    temp["ma20"] = close.rolling(20).mean()
    temp["ma50"] = close.rolling(50).mean()
    temp["golden_cross"] = temp["ma20"] / temp["ma50"]
    
    # RSI
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    temp["RSI"] = 100 - (100 / (1 + gain / loss))
    
    # MACD
    temp["MACD"] = (close.rolling(12).mean() - close.rolling(26).mean()) / close
    
    # Volume
    temp["vol_change"] = vol.pct_change(5)
    temp["vol_vs_avg"] = vol / vol.rolling(21).mean()
    
    # ============================================================
    # 2. Mathematical Features (수학적 피처)
    # ============================================================
    
    # MA slope (1st derivative)
    ma20_slope = temp["ma20"].diff() / close
    temp["ma20_slope"] = ma20_slope
    
    # MA curvature (2nd derivative)
    temp["ma20_curvature"] = ma20_slope.diff() / close
    
    # 2D curvature of (ma20, ma50) trajectory
    ma50_slope = temp["ma50"].diff() / close
    ma50_curv = ma50_slope.diff()
    ma20_curv = ma20_slope.diff()
    num = (ma20_slope * ma50_curv - ma50_slope * ma20_curv).abs()
    den = (ma20_slope**2 + ma50_slope**2)**1.5
    temp["curvature_2d"] = num / (den + 1e-10)
    
    # Cross product of velocity and acceleration vectors
    temp["cross_product_ma"] = ma20_curv - ma20_slope
    
    # Bollinger Band position
    std20 = close.rolling(20).std()
    temp["bollinger"] = (close - temp["ma20"]) / (2 * std20 + 1e-8)
    
    # Relative strength vs sector (섹터 상대 강도)
    # → 계산 후 별도 추가
    
    # OBV (On Balance Volume)
    obv_val = 0
    obv_list = []
    returns = close.pct_change()
    for i in range(len(close)):
        if i == 0:
            obv_list.append(0)
            continue
        if returns.iloc[i] > 0:
            obv_val += vol.iloc[i]
        elif returns.iloc[i] < 0:
            obv_val -= vol.iloc[i]
        obv_list.append(obv_val)
    obv_series = pd.Series(obv_list, index=close.index)
    temp["OBV"] = obv_series / (obv_series.rolling(21).mean().abs() + 1e-8)
    
    # Trading value (거래대금)
    trading_value = close * vol
    temp["trading_value"] = trading_value / (trading_value.rolling(21).mean() + 1e-8)
    
    # Target
    temp["target_short"] = (close.shift(-5) > close).astype(int)
    temp["target_long"] = (close.shift(-21) > close).astype(int)
    temp["ticker"] = ticker
    
    features = pd.concat([features, temp])

features = features.replace([np.inf, -np.inf], np.nan)
ticker_col = features["ticker"].copy()
features = features.groupby("ticker").ffill().bfill()
features["ticker"] = ticker_col
features = features.dropna()

print(f"Features shape: {features.shape}")
print(f"Columns: {features.columns.tolist()}")

# ============================================================
# 3. Feature Correlation Analysis
# ============================================================

feature_list = ["mom1w", "mom1m", "RSI", "MACD", "vol_change", 
                "vol_vs_avg", "ma20_slope", "ma20_curvature",
                "curvature_2d", "cross_product_ma", "bollinger",
                "OBV", "trading_value", "golden_cross"]

corr = features[feature_list].corr()
print("\nCorrelation Matrix:")
print(corr.round(2))

# ============================================================
# 4. Feature Importance Comparison
# ============================================================

train = features[features.index < "2024-01-01"].dropna(subset=feature_list)
test = features[features.index >= "2024-01-01"].dropna(subset=feature_list)

model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
model.fit(train[feature_list], train["target_short"])

print(f"\nAccuracy: {model.score(test[feature_list], test['target_short']):.4f}")

imp = pd.Series(model.feature_importances_, index=feature_list).sort_values(ascending=False)
print("\nFeature Importance:")
print(imp.round(3))

# ============================================================
# 5. Mathematical Features Summary
# ============================================================

print("""
=== Mathematical Features Explored ===

1. MA Curvature (ma20_curvature)
   - 2nd derivative of MA20
   - Detects inflection points
   - Low correlation with MACD (-0.06)

2. 2D Trajectory Curvature (curvature_2d)  
   - Curvature of (MA20, MA50) path in 2D space
   - κ = |MA20' × MA50'' - MA50' × MA20''| / ||(MA20', MA50')||³

3. Cross Product (cross_product_ma)
   - Cross product of velocity and acceleration vectors
   - Trend reversal signal

4. Path Signature
   - Mathematical summary of price path
   - Captures non-linear patterns
   - Requires: pip install esig

5. Bollinger Band Position (bollinger)
   - (price - MA20) / (2 * std)
   - Overbought/oversold signal

6. Sector Relative Strength (rs_1w, rs_1m)
   - Stock momentum - Sector average momentum
   - Cross-sectional signal
""")

[*********************100%***********************]  49 of 50 completed
[*********************100%***********************]  50 of 50 completed
C:\Users\송성원\AppData\Local\Temp\ipykernel_59172\3100791509.py:32: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  temp["mom1w"] = close.pct_change(5)
C:\Users\송성원\AppData\Local\Temp\ipykernel_59172\3100791509.py:33: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  temp["mom1m"] = close.pct_change(21)
C:\Users\송성원\AppData\Local\Temp\ipykernel_59172\3100791509.py:34: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be remov

Features shape: (83650, 20)
Columns: ['mom1w', 'mom1m', 'mom3m', 'ma20', 'ma50', 'golden_cross', 'RSI', 'MACD', 'vol_change', 'vol_vs_avg', 'ma20_slope', 'ma20_curvature', 'curvature_2d', 'cross_product_ma', 'bollinger', 'OBV', 'trading_value', 'target_short', 'target_long', 'ticker']

Correlation Matrix:
                  mom1w  mom1m   RSI  MACD  vol_change  vol_vs_avg  \
mom1w              1.00   0.47  0.51  0.18       -0.02       -0.10   
mom1m              0.47   1.00  0.69  0.79       -0.01       -0.07   
RSI                0.51   0.69  1.00  0.53       -0.01       -0.09   
MACD               0.18   0.79  0.53  1.00       -0.00       -0.05   
vol_change        -0.02  -0.01 -0.01 -0.00        1.00        0.03   
vol_vs_avg        -0.10  -0.07 -0.09 -0.05        0.03        1.00   
ma20_slope         0.45   0.85  0.60  0.91       -0.01       -0.08   
ma20_curvature     0.11  -0.00  0.02  0.02       -0.00       -0.01   
curvature_2d      -0.01  -0.04 -0.05 -0.01       -0.00        0